# Load dan Merge Dataset

In [129]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

In [130]:
df_dokter = pd.read_csv("Jumlah Tenaga Kesehatan Menurut Provinsi, 2025.csv")
df_faskes = pd.read_csv("Jumlah Rumah Sakit Umum, Rumah Sakit Khusus, Puskesmas Rawat Inap, dan Puskesmas Non Rawat Inap Menurut Provinsi, 2025.csv")

In [131]:
print(df_dokter.columns)
print(df_faskes.columns)

Index(['Provinsi', 'Tenaga Medis', 'Tenaga Kebidanan', 'Tenaga Kefarmasian',
       'Tenaga Kesehatan Masyarakat', 'Tenaga Kesehatan Lingkungan',
       'Tenaga Gizi', 'Jumlah Tenaga Medis',
       'Jumlah Tenaga Kesehatan Psikologi Klinis',
       'Jumlah Tenaga Keterapian Fisik', 'Jumlah Tenaga Keteknisan Medis',
       'Jumlah Tenaga Teknik Biomedika',
       'Jumlah Tenaga Kesehatan Tradisional'],
      dtype='object')
Index(['Provinsi', 'Jumlah Rumah Sakit Umum', 'Jumlah Rumah Sakit Khusus',
       'Jumlah Puskesmas Rawat Inap', 'Jumlah Puskesmas Non Rawat Inap'],
      dtype='object')


In [132]:
# Samakan Nama Kolom
df_dokter.rename(columns={"Provinsi": "provinsi"}, inplace=True)
df_faskes.rename(columns={"Provinsi": "provinsi"}, inplace=True)

In [133]:
# Bersihkan nama provinsi (biar cocok)
df_dokter["provinsi"] = df_dokter["provinsi"].str.strip().str.upper()
df_faskes["provinsi"] = df_faskes["provinsi"].str.strip().str.upper()

In [134]:
# Merge
df_merged = pd.merge(df_dokter, df_faskes, on="provinsi", how="inner")

df_merged.head(10)

,provinsi,Tenaga Medis,Tenaga Kebidanan,Tenaga Kefarmasian,Tenaga Kesehatan Masyarakat,Tenaga Kesehatan Lingkungan,Tenaga Gizi,Jumlah Tenaga Medis,Jumlah Tenaga Kesehatan Psikologi Klinis,Jumlah Tenaga Keterapian Fisik,Jumlah Tenaga Keteknisan Medis,Jumlah Tenaga Teknik Biomedika,Jumlah Tenaga Kesehatan Tradisional,Jumlah Rumah Sakit Umum,Jumlah Rumah Sakit Khusus,Jumlah Puskesmas Rawat Inap,Jumlah Puskesmas Non Rawat Inap
0,ACEH,21,21,4,4,1,1,6,77,484,2,2,–,77.0,9,174.0,192.0
1,SUMATERA UTARA,24,25,5,4,986,2,12,45,516,2,4,9,185.0,23,178.0,441.0
2,SUMATERA BARAT,12,9,3,2,641,1,5,31,380,2,2,2,53.0,30,108.0,172.0
3,RIAU,13,10,4,2,487,770,6,60,466,1,2,2,67.0,14,120.0,122.0
4,JAMBI,8,7,2,1,474,521,3,30,218,857,1,9,42.0,3,98.0,110.0
5,SUMATERA SELATAN,17,15,4,3,1,1,6,49,418,2,3,–,72.0,17,110.0,244.0
6,BENGKULU,5,5,1,1,330,578,1,16,86,367,931,–,25.0,3,52.0,127.0
7,LAMPUNG,13,11,3,1,758,806,4,25,327,1,2,56,64.0,19,162.0,160.0
8,KEPULAUAN BANGKA BELITUNG,4,2,1,479,156,247,1,20,114,402,585,–,25.0,4,25.0,39.0
9,KEPULAUAN RIAU,5,3,2,509,342,273,3,25,140,549,805,2,34.0,5,28.0,68.0


# Cleaning Data

In [135]:
df_merged = df_merged.replace("-", 0)

cols_numeric = [
    "Tenaga Medis",
    "Tenaga Kebidanan",
    "Tenaga Kefarmasian",
    "Tenaga Kesehatan Masyarakat",
    "Tenaga Kesehatan Lingkungan",
    "Tenaga Gizi",
    "Jumlah Tenaga Medis",
    "Jumlah Tenaga Kesehatan Psikologi Klinis",
    "Jumlah Tenaga Keterapian Fisik",
    "Jumlah Tenaga Keteknisan Medis",
    "Jumlah Tenaga Teknik Biomedika",
    "Jumlah Tenaga Kesehatan Tradisional",
    "Jumlah Rumah Sakit Umum",
    "Jumlah Rumah Sakit Khusus",
    "Jumlah Puskesmas Rawat Inap",
    "Jumlah Puskesmas Non Rawat Inap"
]

for col in cols_numeric:
    if col in df_merged.columns:
        df_merged[col] = pd.to_numeric(df_merged[col], errors="coerce")

# Fitur Baru

In [136]:
df_merged["total_dokter"] = df_merged["Tenaga Medis"]

df_merged["total_puskesmas"] = (
    df_merged["Jumlah Puskesmas Rawat Inap"] +
    df_merged["Jumlah Puskesmas Non Rawat Inap"]
)

df_merged["total_rumah_sakit"] = (
    df_merged["Jumlah Rumah Sakit Umum"] +
    df_merged["Jumlah Rumah Sakit Khusus"]
)

df_merged["rasio_dokter_puskesmas"] = (
    df_merged["total_dokter"] / df_merged["total_puskesmas"].replace(0, np.nan)
)

df_merged[[
    "provinsi",
    "total_dokter",
    "total_puskesmas",
    "total_rumah_sakit",
    "rasio_dokter_puskesmas"
]].head(10)

,provinsi,total_dokter,total_puskesmas,total_rumah_sakit,rasio_dokter_puskesmas
0,ACEH,21,366.0,86.0,0.057377
1,SUMATERA UTARA,24,619.0,208.0,0.038772
2,SUMATERA BARAT,12,280.0,83.0,0.042857
3,RIAU,13,242.0,81.0,0.053719
4,JAMBI,8,208.0,45.0,0.038462
5,SUMATERA SELATAN,17,354.0,89.0,0.048023
6,BENGKULU,5,179.0,28.0,0.027933
7,LAMPUNG,13,322.0,83.0,0.040373
8,KEPULAUAN BANGKA BELITUNG,4,64.0,29.0,0.062500
9,KEPULAUAN RIAU,5,96.0,39.0,0.052083


In [137]:
df_merged["total_dokter"] = df_merged["total_dokter"].fillna(df_merged["total_dokter"].mean())
df_merged["total_puskesmas"] = df_merged["total_puskesmas"].fillna(df_merged["total_puskesmas"].mean())
df_merged["total_rumah_sakit"] = df_merged["total_rumah_sakit"].fillna(df_merged["total_rumah_sakit"].mean())
df_merged["rasio_dokter_puskesmas"] = df_merged["rasio_dokter_puskesmas"].fillna(df_merged["rasio_dokter_puskesmas"].mean())

# Handle Missing Value

In [138]:
fitur_final = [
    "total_dokter",
    "total_puskesmas",
    "total_rumah_sakit",
    "rasio_dokter_puskesmas",
    "Tenaga Kebidanan",
    "Tenaga Kefarmasian",
    "Tenaga Kesehatan Masyarakat",
    "Tenaga Kesehatan Lingkungan"
]

for col in fitur_final:
    if col in df_merged.columns:
        df_merged[col] = df_merged[col].fillna(df_merged[col].mean())

df_merged[["provinsi"] + fitur_final].head()

,provinsi,total_dokter,total_puskesmas,total_rumah_sakit,rasio_dokter_puskesmas,Tenaga Kebidanan,Tenaga Kefarmasian,Tenaga Kesehatan Masyarakat,Tenaga Kesehatan Lingkungan
0,ACEH,21,366.0,86.0,0.057377,21,4,4,1
1,SUMATERA UTARA,24,619.0,208.0,0.038772,25,5,4,986
2,SUMATERA BARAT,12,280.0,83.0,0.042857,9,3,2,641
3,RIAU,13,242.0,81.0,0.053719,10,4,2,487
4,JAMBI,8,208.0,45.0,0.038462,7,2,1,474


# Augmented berdasarkan data asli 

In [139]:
np.random.seed(42)

df_base = df_merged[["provinsi"] + fitur_final].copy()

cols_to_augment = fitur_final.copy()
n_variasi = 5   # jumlah variasi per provinsi

augmented_rows = []

for _, row in df_base.iterrows():
    for i in range(n_variasi):
        new_row = row.copy()
        
        for col in cols_to_augment:
            # noise 0.85 s.d. 1.15
            noise = np.random.uniform(0.85, 1.15)
            new_row[col] = row[col] * noise
        
        # hitung ulang rasio supaya konsisten
        new_row["rasio_dokter_puskesmas"] = (
            new_row["total_dokter"] / new_row["total_puskesmas"]
            if new_row["total_puskesmas"] != 0 else 0
        )
        
        new_row["provinsi"] = f"{row['provinsi']}_VAR{i+1}"
        augmented_rows.append(new_row)

df_augmented = pd.DataFrame(augmented_rows)

# gabungkan data asli + augmented
df_final = pd.concat([df_base, df_augmented], ignore_index=True)

df_final

,provinsi,total_dokter,total_puskesmas,total_rumah_sakit,rasio_dokter_puskesmas,Tenaga Kebidanan,Tenaga Kefarmasian,Tenaga Kesehatan Masyarakat,Tenaga Kesehatan Lingkungan
0,ACEH,21.000000,366.000000,86.000000,0.057377,21.000000,4.000000,4.000000,1.000000
1,SUMATERA UTARA,24.000000,619.000000,208.000000,0.038772,25.000000,5.000000,4.000000,986.000000
2,SUMATERA BARAT,12.000000,280.000000,83.000000,0.042857,9.000000,3.000000,2.000000,641.000000
3,RIAU,13.000000,242.000000,81.000000,0.053719,10.000000,4.000000,2.000000,487.000000
4,JAMBI,8.000000,208.000000,45.000000,0.038462,7.000000,2.000000,1.000000,474.000000
...,...,...,...,...,...,...,...,...,...
229,INDONESIA_VAR1,506.090767,11.642407,493.766917,43.469598,332.584350,152.960440,60.805022,23.554675
230,INDONESIA_VAR2,635.999230,10.421008,487.516844,61.030488,355.658298,153.744888,63.314207,22.164146
231,INDONESIA_VAR3,527.233021,10.859654,467.383068,48.549706,354.946435,145.540408,66.312588,23.174221
232,INDONESIA_VAR4,663.965910,10.249156,564.208331,64.782496,288.249936,171.590993,73.802365,25.762089


# Normalisasi

In [143]:
# fitur yang di-scale
features_scale = [
    "total_dokter",
    "total_puskesmas",
    "total_rumah_sakit",
    "Tenaga Kebidanan",
    "Tenaga Kefarmasian",
    "Tenaga Kesehatan Masyarakat",
    "Tenaga Kesehatan Lingkungan"
]

# fitur final untuk content-based
cb_features = features_scale + ["rasio_dokter_puskesmas"]

df_model_scaled = df_final.copy()

# scale hanya fitur tertentu
df_model_scaled[features_scale] = scaler.fit_transform(df_final[features_scale])

# rasio tetap asli
df_model_scaled["rasio_dokter_puskesmas"] = df_final["rasio_dokter_puskesmas"]

# tampilkan
df_model_scaled[["provinsi"] + cb_features].head()

,provinsi,total_dokter,total_puskesmas,total_rumah_sakit,Tenaga Kebidanan,Tenaga Kefarmasian,Tenaga Kesehatan Masyarakat,Tenaga Kesehatan Lingkungan,rasio_dokter_puskesmas
0,ACEH,0.030220,0.282967,0.131144,0.019368,0.003028,0.003020,0.000129,0.057377
1,SUMATERA UTARA,0.034745,0.484205,0.352806,0.023215,0.004000,0.003020,0.887569,0.038772
2,SUMATERA BARAT,0.016645,0.214562,0.125694,0.007828,0.002056,0.001097,0.576740,0.042857
3,RIAU,0.018154,0.184336,0.122060,0.008790,0.003028,0.001097,0.437993,0.053719
4,JAMBI,0.010612,0.157292,0.056652,0.005905,0.001084,0.000136,0.426280,0.038462


# Content Based Filtering

Hitung Similarity

In [144]:
X_cb = df_model_scaled[cb_features].fillna(0)

similarity_matrix = cosine_similarity(X_cb)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=df_model_scaled["provinsi"],
    columns=df_model_scaled["provinsi"]
)

print(similarity_df.iloc[:5, :5])

provinsi            ACEH  SUMATERA UTARA  SUMATERA BARAT      RIAU     JAMBI
provinsi                                                                    
ACEH            1.000000        0.546773        0.399955  0.456913  0.372423
SUMATERA UTARA  0.546773        1.000000        0.981033  0.989181  0.966671
SUMATERA BARAT  0.399955        0.981033        1.000000  0.997020  0.996889
RIAU            0.456913        0.989181        0.997020  1.000000  0.990593
JAMBI           0.372423        0.966671        0.996889  0.990593  1.000000


Fungsi Rekomendasi

In [151]:
def get_similar_real_provinces(provinsi, top_n=5):
    similar_scores = similarity_df[provinsi].sort_values(ascending=False)

    base_name = provinsi.split("_VAR")[0]

    # ambil hanya provinsi asli, buang semua _VAR
    mask_real_only = ~similar_scores.index.str.contains("_VAR")
    mask_not_self = similar_scores.index != base_name

    filtered_scores = similar_scores[mask_real_only & mask_not_self].head(top_n)

    result = pd.DataFrame({
        "provinsi_mirip": filtered_scores.index,
        "skor_similarity": filtered_scores.values
    })

    return result

Uji

In [154]:
get_similar_real_provinces("ACEH", top_n=10)

,provinsi_mirip,skor_similarity
0,SUMATERA SELATAN,0.998959
1,SULAWESI SELATAN,0.996824
2,NUSA TENGGARA TIMUR,0.983559
3,JAWA BARAT,0.951790
4,JAWA TENGAH,0.946145
5,JAWA TIMUR,0.931750
6,SUMATERA UTARA,0.546773
7,BANTEN,0.488811
8,RIAU,0.456913
9,KALIMANTAN TIMUR,0.447907


# Konwledge Based Filtering

In [155]:
# ==============================
# 1. AMBIL DATA ASLI SAJA
# ==============================
df_kb = df_final[~df_final["provinsi"].str.contains("_VAR")].copy()

# ==============================
# 2. PILIH FITUR UNTUK SCORING
# ==============================
kb_features = [
    "total_dokter",
    "total_puskesmas",
    "total_rumah_sakit",
    "rasio_dokter_puskesmas"
]

# ==============================
# 3. PASTIKAN TIDAK ADA NaN
# ==============================
for col in kb_features:
    df_kb[col] = pd.to_numeric(df_kb[col], errors="coerce")
    df_kb[col] = df_kb[col].fillna(df_kb[col].mean())

# ==============================
# 4. NORMALISASI UNTUK SCORING
# ==============================
scaler_kb = MinMaxScaler()

df_kb["dokter_scaled"] = scaler_kb.fit_transform(df_kb[["total_dokter"]])
df_kb["puskesmas_scaled"] = scaler_kb.fit_transform(df_kb[["total_puskesmas"]])
df_kb["rs_scaled"] = scaler_kb.fit_transform(df_kb[["total_rumah_sakit"]])
df_kb["rasio_scaled"] = scaler_kb.fit_transform(df_kb[["rasio_dokter_puskesmas"]])

# ==============================
# 5. HITUNG SKOR PRIORITAS
# ==============================
df_kb["skor_prioritas"] = (
    0.40 * (1 - df_kb["rasio_scaled"]) +   # rasio kecil = prioritas tinggi
    0.25 * df_kb["puskesmas_scaled"] +     # puskesmas banyak = prioritas tinggi
    0.20 * (1 - df_kb["dokter_scaled"]) +  # dokter sedikit = prioritas tinggi
    0.15 * (1 - df_kb["rs_scaled"])        # RS sedikit = prioritas tinggi
)

# ==============================
# 6. URUTKAN DARI PALING PRIORITAS
# ==============================
df_kb = df_kb.sort_values("skor_prioritas", ascending=False)

# ==============================
# 7. TAMPILKAN HASIL
# ==============================
df_kb[[
    "provinsi",
    "total_dokter",
    "total_puskesmas",
    "total_rumah_sakit",
    "rasio_dokter_puskesmas",
    "skor_prioritas"
]].head(10)

,provinsi,total_dokter,total_puskesmas,total_rumah_sakit,rasio_dokter_puskesmas,skor_prioritas
11,JAWA BARAT,74.0,1106.0,436.0,0.066908,0.848674
18,NUSA TENGGARA TIMUR,16.0,441.0,67.0,0.036281,0.827622
1,SUMATERA UTARA,24.0,619.0,208.0,0.038772,0.823233
12,JAWA TENGAH,66.0,881.0,369.0,0.074915,0.820098
14,JAWA TIMUR,74.0,976.0,442.0,0.075820,0.817152
26,SULAWESI SELATAN,27.0,474.0,124.0,0.056962,0.814151
27,SULAWESI TENGGARA,10.0,308.0,43.0,0.032468,0.806553
0,ACEH,21.0,366.0,86.0,0.057377,0.802951
5,SUMATERA SELATAN,17.0,354.0,89.0,0.048023,0.800755
7,LAMPUNG,13.0,322.0,83.0,0.040373,0.796679


Fungsi Rekomendasi

In [158]:
def get_priority_provinces(top_n=10):
    return df_kb[[
        "provinsi",
        "total_dokter",
        "total_puskesmas",
        "total_rumah_sakit",
        "rasio_dokter_puskesmas",
        "skor_prioritas"
    ]].head(top_n)

In [159]:
get_priority_provinces(10)

,provinsi,total_dokter,total_puskesmas,total_rumah_sakit,rasio_dokter_puskesmas,skor_prioritas
11,JAWA BARAT,74.0,1106.0,436.0,0.066908,0.848674
18,NUSA TENGGARA TIMUR,16.0,441.0,67.0,0.036281,0.827622
1,SUMATERA UTARA,24.0,619.0,208.0,0.038772,0.823233
12,JAWA TENGAH,66.0,881.0,369.0,0.074915,0.820098
14,JAWA TIMUR,74.0,976.0,442.0,0.075820,0.817152
26,SULAWESI SELATAN,27.0,474.0,124.0,0.056962,0.814151
27,SULAWESI TENGGARA,10.0,308.0,43.0,0.032468,0.806553
0,ACEH,21.0,366.0,86.0,0.057377,0.802951
5,SUMATERA SELATAN,17.0,354.0,89.0,0.048023,0.800755
7,LAMPUNG,13.0,322.0,83.0,0.040373,0.796679


# Ensembling Content Based dan Knowledge Based

In [160]:
def hybrid_recommendation(provinsi, top_n=5, alpha=0.6, beta=0.4):
    
    # ==============================
    # 1. Ambil similarity
    # ==============================
    sim_scores = similarity_df[provinsi].sort_values(ascending=False)
    
    # ambil hanya provinsi asli & bukan dirinya
    base_name = provinsi.split("_VAR")[0]
    
    sim_scores = sim_scores[
        (~sim_scores.index.str.contains("_VAR")) &
        (sim_scores.index != base_name)
    ]
    
    # ubah jadi dataframe
    df_sim = pd.DataFrame({
        "provinsi": sim_scores.index,
        "similarity": sim_scores.values
    })
    
    # ==============================
    # 2. Ambil skor prioritas
    # ==============================
    df_priority = df_kb[[
        "provinsi",
        "skor_prioritas"
    ]]
    
    # ==============================
    # 3. Merge keduanya
    # ==============================
    df_hybrid = pd.merge(df_sim, df_priority, on="provinsi")
    
    # ==============================
    # 4. Hitung skor hybrid
    # ==============================
    df_hybrid["skor_hybrid"] = (
        alpha * df_hybrid["similarity"] +
        beta * df_hybrid["skor_prioritas"]
    )
    
    # ==============================
    # 5. Ranking
    # ==============================
    df_hybrid = df_hybrid.sort_values("skor_hybrid", ascending=False)
    
    return df_hybrid.head(top_n)

In [161]:
hybrid_recommendation("ACEH", top_n=5)

,provinsi,similarity,skor_prioritas,skor_hybrid
1,SULAWESI SELATAN,0.996824,0.814151,0.923755
2,NUSA TENGGARA TIMUR,0.983559,0.827622,0.921184
0,SUMATERA SELATAN,0.998959,0.800755,0.919677
3,JAWA BARAT,0.951790,0.848674,0.910543
4,JAWA TENGAH,0.946145,0.820098,0.895726


In [162]:
get_priority_provinces(10)

,provinsi,total_dokter,total_puskesmas,total_rumah_sakit,rasio_dokter_puskesmas,skor_prioritas
11,JAWA BARAT,74.0,1106.0,436.0,0.066908,0.848674
18,NUSA TENGGARA TIMUR,16.0,441.0,67.0,0.036281,0.827622
1,SUMATERA UTARA,24.0,619.0,208.0,0.038772,0.823233
12,JAWA TENGAH,66.0,881.0,369.0,0.074915,0.820098
14,JAWA TIMUR,74.0,976.0,442.0,0.075820,0.817152
26,SULAWESI SELATAN,27.0,474.0,124.0,0.056962,0.814151
27,SULAWESI TENGGARA,10.0,308.0,43.0,0.032468,0.806553
0,ACEH,21.0,366.0,86.0,0.057377,0.802951
5,SUMATERA SELATAN,17.0,354.0,89.0,0.048023,0.800755
7,LAMPUNG,13.0,322.0,83.0,0.040373,0.796679
